# OpenDistillation v0 Demo Skeleton

> Upload docs. Distill a tiny local model. Run it locally.

This notebook is the first runnable skeleton for the OpenDistillation v0 flow. It covers Milestones 1-3 only: text upload/loading, validation, chunking, dataset schema validation, and deterministic mock teacher generation.

It does not train a model, download a model, use a GPU, call paid APIs, or export GGUF files yet.

## Runtime setup

Run this notebook from the repository root for now. In Colab, the future public GitHub notebook link will clone or install the package once the remote exists. The skeleton uses only Python standard-library code and local helper modules.

In [ ]:
from pathlib import Path
import json
import sys
import tempfile

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src" / "opendistillation").exists() and (PROJECT_ROOT.parent / "src" / "opendistillation").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

src_path = PROJECT_ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from opendistillation import (
    MockTeacherEngine,
    TeacherRequest,
    chunk_text,
    load_text_document,
    rows_to_jsonl,
)

print(f"Using project root: {PROJECT_ROOT}")
print("Runtime: CPU-only skeleton; no model downloads or training calls will run.")

## Upload or load a TXT/MD file

In Colab, this cell offers a file upload. Outside Colab, it uses `examples/sample-notes.md` so the notebook can run top to bottom without any external service.

In [ ]:
def load_uploaded_or_sample():
    try:
        from google.colab import files  # type: ignore
    except ImportError:
        sample_path = PROJECT_ROOT / "examples" / "sample-notes.md"
        return sample_path.name, sample_path.read_text(encoding="utf-8")

    uploaded = files.upload()
    if not uploaded:
        raise ValueError("No file uploaded. Upload one .txt or .md file to continue.")
    filename, content = next(iter(uploaded.items()))
    return filename, content


filename, content = load_uploaded_or_sample()
document = load_text_document(filename, content)

print(f"File: {document.filename}")
print(f"Extension: {document.extension}")
print(f"Characters: {document.char_count}")
print(f"Approx. words: {document.word_count}")
if document.warnings:
    print("Warnings:")
    for warning in document.warnings:
        print(f"- {warning}")
print("\nPreview:\n")
print(document.preview)

## Chunk the document

The v0 chunker prefers paragraph boundaries, preserves source order, removes empty chunks, and assigns stable IDs like `chunk-0001`.

In [ ]:
chunks = chunk_text(document.text, max_chars=300)
print(f"Chunks: {len(chunks)}")

for chunk in chunks[:3]:
    print("=" * 72)
    print(f"{chunk.id} | chars={chunk.char_count} | words={chunk.word_count}")
    print(chunk.text[:500])

## Generate mock training examples

This skeleton uses `MockTeacherEngine`, a deterministic local teacher path. It does not send text to a remote endpoint. Later goals can replace this engine with real open-source teacher backends while keeping the same notebook flow.

In [ ]:
teacher_engine = MockTeacherEngine()
request = TeacherRequest(chunks=chunks, examples_per_chunk=2)
rows = teacher_engine.generate(request)
dataset_jsonl = rows_to_jsonl(rows)

print(f"Teacher engine: {teacher_engine.name}")
print(f"Sends text to remote endpoint: {teacher_engine.sends_data_remote}")
print(f"Generated examples: {len(rows)}")
print("\nFirst 5 examples:\n")
for row in rows[:5]:
    print(json.dumps(row, ensure_ascii=False, indent=2))

## Dataset JSONL preview

The initial schema is one JSON object per line with exactly these fields: `instruction`, `response`, and `source_chunk_id`.

In [ ]:
print("First JSONL lines:\n")
print("\n".join(dataset_jsonl.splitlines()[:5]))

# Save to the runtime temp directory, not the repository, so generated data is not committed.
output_path = Path(tempfile.gettempdir()) / "opendistillation_mock_training_data.jsonl"
output_path.write_text(dataset_jsonl, encoding="utf-8")
print(f"Saved runtime dataset to {output_path}")

try:
    from google.colab import files  # type: ignore
except ImportError:
    print("Download helper is available only in Colab; use the path above locally.")
else:
    files.download(str(output_path))

## Training placeholder

Real student fine-tuning is intentionally not implemented in this skeleton. Later milestones can plug in a `TrainingEngine` behind this point using open-source tools such as Hugging Face Transformers, PEFT/LoRA, TRL, or Unsloth.

The notebook flow should stay the same: validated dataset in, trained adapter/model output out.

In [ ]:
print("Training placeholder: skipped.")
print("No GPU, model download, model training, or paid API call is used in this skeleton.")

## Export placeholder

GGUF export and local runtime instructions are later milestones. The intended `ExportEngine` plug-in point is after training output exists: convert or merge the model output, then document llama.cpp and/or Ollama-style local commands.

In [ ]:
print("Export placeholder: skipped.")
print("No GGUF files, model artifacts, or local runtime files are created by this skeleton.")